In [1]:
"""
过敏状态转移模型演示
"""

import sys
import os
from pathlib import Path

# 获取当前 notebook 所在的目录（Jupyter 中）
try:
    # 尝试获取 __file__（在 .py 文件中有效）
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # 在 Jupyter notebook 中，使用当前工作目录
    current_dir = os.getcwd()
    
# 将当前目录添加到系统路径
sys.path.append(current_dir)

from models.allergy_model import AllergyStateTransitionModel
from models.base_model import EventType

TabError: inconsistent use of tabs and spaces in indentation (allergy_model.py, line 138)

In [ ]:
def demo_allergy_model():
    """演示过敏状态转移模型"""
    print("=" * 80)
    print("过敏状态转移模型演示")
    print("=" * 80)
    
    # 创建模型实例 - k=3, n=4
    model = AllergyStateTransitionModel(k=3, n=4)
    
    k_val = model.model_params['k']
    n_val = model.model_params['n']
    
    print(f"\n📊 模型信息:")
    print(f"   参数: k={k_val}, n={n_val}")
    print(f"   初始状态: (0,0,1) 不健康")
    print(f"   可用干预:")
    print(f"      - 干预A (过敏原): (1,1,0) 🔴")
    print(f"      - 干预B (治疗): (1,0,0) 🟢")
    print(f"      - 无干预: (0,0,0) ⚪")
    
    print(f"\n📖 状态说明:")
    print(f"   (0,0,1) → 不健康状态")
    print(f"   (0,0,0) → 健康状态")
    print(f"   (1,1,0) → 不利状态（过敏反应）")
    
    print(f"\n📖 事件说明:")
    print(f"   E6a (不良反应): 最近 {k_val} 步内出现干预A → 不利状态 (1,1,0)")
    print(f"   E6b (完全恢复): 连续 {n_val} 次干预B → 健康 (0,0,0)")
    print(f"   E6c (无有效转移): 其他情况 → 不健康 (0,0,1)")
    
    # 场景1: 不良反应 (单次干预A)
    print("\n" + "=" * 80)
    print(f"场景1: 不良反应 - 单次干预A")
    print("=" * 80)
    
    interventions_scene1 = [(1, 1, 0)]  # 一次干预A
    
    result1 = model.simulate(interventions_scene1, return_details=True)
    _print_results(result1, model, title="单次干预A → 不利状态，持续k步")
    
    # 场景2: 完全恢复 (连续n次干预B)
    print("\n" + "=" * 80)
    print(f"场景2: 完全恢复 - 连续 {n_val} 次干预B")
    print("=" * 80)
    
    interventions_scene2 = [(1, 0, 0)] * n_val
    
    result2 = model.simulate(interventions_scene2, return_details=True)
    _print_results(result2, model, title=f"连续{n_val}次干预B → 健康")
    
    # 场景3: 不良反应后自然恢复
    print("\n" + "=" * 80)
    print(f"场景3: 不良反应后自然恢复 - 干预A后等待{k_val}步")
    print("=" * 80)
    
    interventions_scene3 = [(1, 1, 0)] + [(0, 0, 0)] * (k_val + 2)
    
    result3 = model.simulate(interventions_scene3, return_details=True)
    _print_results(result3, model, title=f"干预A后，不利状态持续{k_val}步，然后恢复")
    
    # 场景4: 不良反应后使用干预B治疗
    print("\n" + "=" * 80)
    print(f"场景4: 不良反应后使用干预B治疗")
    print("=" * 80)
    
    interventions_scene4 = (
        [(1, 1, 0)] +           # 干预A，引起过敏
        [(1, 0, 0)] * n_val     # 干预B，治疗恢复
    )
    
    result4 = model.simulate(interventions_scene4, return_details=True)
    _print_results(result4, model, title="不良反应后使用干预B → 恢复健康")
    
    # 场景5: 多次过敏反应
    print("\n" + "=" * 80)
    print(f"场景5: 多次过敏反应 - 反复接触过敏原")
    print("=" * 80)
    
    interventions_scene5 = [
        (1, 1, 0),  # t=1: 干预A
        (0, 0, 0),  # t=2: 无干预
        (0, 0, 0),  # t=3: 无干预
        (1, 1, 0),  # t=4: 干预A（再次接触）
        (0, 0, 0),  # t=5: 无干预
    ]
    
    result5 = model.simulate(interventions_scene5, return_details=True)
    _print_results(result5, model, title="多次接触过敏原 → 反复进入不利状态")


def demo_allergy_cycle():
    """演示过敏循环：接触过敏原 → 不利状态 → 恢复"""
    print("\n" + "=" * 80)
    print("过敏循环演示：接触过敏原 → 不利状态 → 恢复")
    print("=" * 80)
    
    model = AllergyStateTransitionModel(k=3, n=4)
    k_val = model.model_params['k']
    
    # 创建完整的过敏循环
    interventions = [
        (1, 1, 0),                      # 接触过敏原
    ] + [(0, 0, 0)] * k_val + [         # 等待不利状态结束
        (1, 1, 0),                      # 再次接触过敏原
    ]
    
    result = model.simulate(interventions, return_details=True)
    
    print(f"\n干预序列:")
    for t, inter in enumerate(interventions, 1):
        if inter == model.INTERVENTION_A:
            print(f"   t={t:2d}: {inter} 🔴 干预A (过敏原)")
        elif inter == model.INTERVENTION_B:
            print(f"   t={t:2d}: {inter} 🟢 干预B (治疗)")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 80)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<40}")
    print("-" * 80)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        
        if state == model.HEALTHY:
            state_name = "健康"
        elif state == model.ADVERSE:
            state_name = "不利状态"
        else:
            state_name = "不健康"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.ADVERSE_REACTION:
                event_str = "E6a (不良反应) ⚡"
            elif event == EventType.COMPLETE_TRANSITION:
                event_str = "E6b (完全恢复) ⭐"
            else:
                event_str = "E6c (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<40}")
    
    print(f"\n💡 观察:")
    print(f"   - 接触干预A后，立即进入不利状态")
    print(f"   - 不利状态持续 {k_val} 步")
    print(f"   - 不利状态结束后自动恢复到不健康状态")
    print(f"   - 在不利状态期间再次接触过敏原会重置计数器")


def demo_treatment_comparison():
    """治疗对比演示"""
    print("\n" + "=" * 80)
    print("治疗对比：干预B vs 无干预")
    print("=" * 80)
    
    model = AllergyStateTransitionModel(k=3, n=4)
    n_val = model.model_params['n']
    
    print(f"\n场景: 接触过敏原后，对比有/无干预B的效果")
    print("-" * 60)
    
    # 无治疗
    interventions_no_treatment = [(1, 1, 0)] + [(0, 0, 0)] * 5
    result_no = model.simulate(interventions_no_treatment)
    
    # 有治疗
    model.reset_internal_state()
    interventions_treatment = [(1, 1, 0)] + [(1, 0, 0)] * n_val
    result_treatment = model.simulate(interventions_treatment)
    
    print(f"\n{'治疗方案':<15} {'最终状态':<20} {'是否健康':<10}")
    print("-" * 60)
    
    final_no = result_no["states"][-1]
    final_treatment = result_treatment["states"][-1]
    
    no_healthy = "是" if final_no == model.HEALTHY else "否"
    treatment_healthy = "是" if final_treatment == model.HEALTHY else "否"
    
    print(f"{'无治疗 (等待)':<15} {str(final_no):<20} {no_healthy:<10}")
    print(f"{f'干预B ({n_val}次)':<15} {str(final_treatment):<20} {treatment_healthy:<10}")
    
    print(f"\n💡 结论:")
    print(f"   - 无治疗：不利状态结束后恢复到不健康 (0,0,1)")
    print(f"   - 干预B治疗：连续{n_val}次干预B后恢复到健康 (0,0,0)")


def _print_results(result: dict, model, title: str = ""):
    """打印模拟结果"""
    if title:
        print(f"\n{title}")
    
    print(f"\n干预序列:")
    for t, inter in enumerate(result["interventions"], 1):
        if inter == model.INTERVENTION_A:
            print(f"   t={t:2d}: {inter} 🔴 干预A (过敏原)")
        elif inter == model.INTERVENTION_B:
            print(f"   t={t:2d}: {inter} 🟢 干预B (治疗)")
        else:
            print(f"   t={t:2d}: {inter} ⚪ 无干预")
    
    print(f"\n状态演变:")
    print("-" * 80)
    print(f"{'步数':<6} {'可观测状态':<20} {'状态说明':<15} {'事件':<40}")
    print("-" * 80)
    
    for i in range(len(result["states"])):
        state = result["states"][i]
        
        if state == model.HEALTHY:
            state_name = "健康"
        elif state == model.ADVERSE:
            state_name = "不利状态"
        else:
            state_name = "不健康"
        
        if i == 0:
            event_str = "-"
        else:
            event = result["events"][i-1]
            if event == EventType.ADVERSE_REACTION:
                event_str = "E6a (不良反应) ⚡"
            elif event == EventType.COMPLETE_TRANSITION:
                event_str = "E6b (完全恢复) ⭐"
            else:
                event_str = "E6c (无有效转移)"
        
        display_state = f"{state} ({state_name})"
        print(f"{i:<6} {display_state:<20} {state_name:<15} {event_str:<40}")
    
    # 显示内部状态
    internal = model.get_internal_state()
    if internal["is_in_adverse"]:
        print(f"\n⚠️  当前处于不利状态，剩余 {internal['adverse_counter']} 步")


In [ ]:
if __name__ == "__main__":
    demo_allergy_model()
    demo_allergy_cycle()
    demo_treatment_comparison()